# Hybrid MFUSG grids with circle-barrier holes

This example uses the exact-barrier implementation from `feature/exact-barrier-faces`
(PR #73). A closed barrier follows the requested buffer perimeter. Its whole interior
Voronoi cells are removed, and a second Gmsh pass fills that hole with retained triangles.
Exterior cell centres and geometry remain unchanged apart from coordinate roundoff.
There is no hybrid-specific generator offset, corner repair or cell trimming.

The first construction pass currently meshes both sides of the perimeter; its interior
cells are support geometry, excluded from the final model. The buffer does not assign
hydraulic resistance. Geological zones remain independent.

## Setup

From this checkout, install the optional examples dependencies with
`python -m pip install -e ".[examples]"`. Install an MFUSG executable separately
(for example `python -m flopy.utils.get_modflow /path/to/bin --subset mfusg` in a terminal).
Put it on `PATH` or set the `MFUSG_EXE` environment variable. No downloads occur here.


In [ ]:
import os
from pathlib import Path
import shutil
import tempfile

import flopy
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import numpy as np
from shapely.geometry import LineString, box
from shapely.ops import unary_union

from vorflow import ConceptualMesh, MeshGenerator, VoronoiTessellator, set_verbosity

set_verbosity(0)
executable = os.environ.get("MFUSG_EXE") or shutil.which("mfusg")
if not executable:
    raise FileNotFoundError("Install mfusg and add it to PATH, or set MFUSG_EXE to its executable path.")


## 1. Define the faults and sharp buffer footprints

For `h = 5 m`, the buffer radius is `sqrt(3)*h = 8.66 m` (full straight corridor
width `17.32 m`). This is two equilateral-triangle heights on each side of the fault,
an illustrative width rather than a triangle-quality guarantee. Square caps and
mitred joins retain sharp corners. The realised footprint must match this request.


In [ ]:
h = 5.0
domain = box(0, 0, 200, 100)
faults = {
    "zigzag": [LineString([(55, 40), (80, 60), (105, 40), (145, 60)])],
    "junction": [LineString([(55, 50), (100, 50), (145, 50)]),
                 LineString([(100, 50), (125, 75)])],
}
models = {}
for name, lines in faults.items():
    footprint = unary_union(lines).buffer(np.sqrt(3) * h, cap_style="square", join_style="mitre")
    cm = ConceptualMesh()
    cm.add_polygon(domain, zone_id="aquifer", resolution=10.0)
    cm.add_polygon(footprint, resolution=h, subregion="triangle")
    for number, fault in enumerate(lines):
        cm.add_line(fault, line_id=f"fault_{number}", resolution=h, is_barrier=True)
    mg = MeshGenerator(background_lc=10.0, conceptual_mesh=cm)
    mg.generate()
    vt = VoronoiTessellator(mg, cm)
    grid = vt.generate()
    models[name] = dict(cm=cm, mg=mg, vt=vt, grid=grid, footprint=footprint, faults=lines)


## 2. Check whole cells, coverage and corners

The reservation perimeter goes through the standard circle-barrier planner before
Gmsh. Every retained exterior cell has the same numerical centre and footprint as
its cell in `construction_grid`. `hybrid_report` lists the complete cells removed
from the hole. The triangle pass matches the resulting interface edges.

The close-ups show final Voronoi generators (dots) and the standard barrier circles.
They are not an overlay of the original Gmsh elements.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for row, (name, entry) in enumerate(models.items()):
    grid, vt, mg = entry["grid"], entry["vt"], entry["mg"]
    exterior = grid[grid.cell_type == "voronoi"]
    triangles = grid[grid.cell_type == "triangle"]
    before = vt.construction_grid.set_index("node_id").loc[exterior.node_id]
    assert np.array_equal(exterior[["x", "y"]].to_numpy(), before[["x", "y"]].to_numpy())
    assert max(a.symmetric_difference(b).area for a, b in zip(exterior.geometry, before.geometry)) < 1e-7
    assert vt.hybrid_report["cell_clip_area"] == 0
    assert vt.n_exact_barrier_cuts == vt.n_exact_barrier_mirrors == vt.n_barrier_mirrors == 0
    combined = unary_union(grid.geometry)
    assert combined.symmetric_difference(domain).area < 1e-7
    assert abs(grid.area.sum() - combined.area) < 1e-7
    assert (exterior.geometry.convex_hull.area - exterior.area).max() < 1e-7
    assert triangles.minimum_angle.min() >= 20
    for col, ax in enumerate(axes[row]):
        exterior.plot(ax=ax, facecolor="#d4e9f8", edgecolor="#526b7a", linewidth=0.5)
        triangles.plot(ax=ax, facecolor="#f6d9a8", edgecolor="#9b5725", linewidth=0.5)
        ax.plot(*entry["footprint"].exterior.xy, color="#bf2547", linewidth=1.3)
        for fault in entry["faults"]:
            ax.plot(*fault.xy, color="black", linewidth=1.3)
        ax.set(aspect="equal", xlabel="x (m)", ylabel="y (m)")
        if col:
            centre = (80, 71.1) if name == "zigzag" else (120.9, 58.7)
            ax.set(xlim=(centre[0]-10, centre[0]+10), ylim=(centre[1]-10, centre[1]+10), title=f"{name}: whole cells at a corner")
            ax.scatter(exterior.x, exterior.y, s=12, color="#155aab", label="Voronoi generators")
            centres, radii = mg.barrier_circles
            for xy, radius in zip(centres, radii):
                if np.linalg.norm(xy-centre) < 18:
                    ax.add_patch(Circle(xy, radius, fill=False, color="#9b5725", alpha=.35, linewidth=.7))
        else:
            ax.set_title(f"{name}: closed barrier hole filled with triangles")
    print(f"{name}: {len(exterior)} whole exterior cells, {len(triangles)} triangles; "
          f"minimum angle {triangles.minimum_angle.min():.2f} degrees; "
          f"{vt.hybrid_report['removed_cells']} whole support cells removed")
plt.tight_layout()
plt.show()


## 3. Set up and run one confined MFUSG layer

Top is 10 m, bottom 0 m and uniform conductivity 10 m/day. The left and right
boundary generator rows have fixed heads of 20 and 10 m; other boundaries are
no-flow. Initial interior heads are 15 m, so the solver must recover the solution.

The default temporary workspace is removed when its block ends. To retain exported
DISU, GSF and model output, replace `TemporaryDirectory` with your own `Path`, create
it with `mkdir`, and use that folder as `model_ws`.


In [ ]:
for name, entry in models.items():
    grid, vt = entry["grid"], entry["vt"]
    with tempfile.TemporaryDirectory(prefix=f"vorflow-{name}-") as folder:
        workspace = Path(folder)
        model = flopy.mfusg.MfUsg(modelname=name, structured=False, model_ws=workspace, exe_name=executable)
        vt.to_mfusgdisu(model, top=10.0, botm=[0.0], nlay=1, nper=1, steady=True)
        vt.to_gsf(workspace / f"{name}.gsf", top=10.0, botm=[0.0], nlay=1)
        ibound = np.ones(len(grid), dtype=int)
        left, right = np.isclose(grid.x, 0), np.isclose(grid.x, 200)
        ibound[left | right] = -1
        initial = np.full(len(grid), 15.0)
        initial[left], initial[right] = 20.0, 10.0
        flopy.mfusg.MfUsgBas(model, ibound=ibound, strt=initial)
        flopy.mfusg.MfUsgLpf(model, laytyp=0, hk=10.0, vka=10.0, ipakcb=53)
        flopy.mfusg.MfUsgSms(model, nonlinmeth=0, linmeth=2, clin="cg", hclose=1e-8,
                           hiclose=1e-8, rclosepcgu=1e-8, mxiter=100, iter1=500)
        flopy.modflow.ModflowOc(model, stress_period_data={(0, 0): ["save head", "save budget", "print budget"]}, compact=True)
        model.write_input()
        success, report = model.run_model(silent=True, report=True)
        assert success and any("Normal termination" in line for line in report)
        heads = flopy.utils.HeadUFile(workspace / f"{name}.hds").get_data()[0].ravel()
        budget, _ = flopy.utils.MfusgListBudget(workspace / f"{name}.list").get_budget()
        entry["heads"] = heads.copy()
        entry["flow"] = float(budget["CONSTANT_HEAD_IN"].max())
        entry["budget_discrepancy"] = float(np.abs(budget["PERCENT_DISCREPANCY"]).max())


## 4. Check analytical flow and plot heads on the original cells

The reference flow is `K * width * thickness * head_difference / length = 500 m³/day`.
Require finite heads between 10 and 20 m, water-budget discrepancy below 0.1%,
and physical flow within 2% of this reference. These checks do not establish
accuracy for anisotropy, heterogeneous properties or hydraulic barriers.

One triangle reservation is supported. Embedded points and non-barrier lines in
the reservation, exact alignment of other polygon boundaries, hydraulic resistance
and ghost-node correction terms are deferred. Unresolved perimeter alignment fails
explicitly rather than clipping a cell or moving the requested boundary.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7))
for ax, (name, entry) in zip(axes, models.items()):
    heads, grid = entry["heads"], entry["grid"]
    assert np.isfinite(heads).all() and heads.min() >= 10-1e-6 and heads.max() <= 20+1e-6
    assert abs(entry["flow"] / 500 - 1) < .02
    assert entry["budget_discrepancy"] < .1
    exact = 20 - grid.x.to_numpy() / 20
    assert np.max(np.abs(heads-exact)) < .1
    grid.assign(head=heads).plot(column="head", ax=ax, cmap="viridis", vmin=10, vmax=20,
                               edgecolor="0.3", linewidth=.25, legend=True)
    ax.set(aspect="equal", xlabel="x (m)", ylabel="y (m)", title=f"{name}: MFUSG head (m)")
    print(f"{name}: normal termination; flow {entry['flow']:.3f} m3/day; "
          f"budget discrepancy {entry['budget_discrepancy']:.5f}%")
plt.tight_layout()
plt.show()
